# Azure Voicebot Hackathon Smoke Test

This notebook validates:
- Azure OpenAI `gpt-5.1`
- Azure OpenAI `gpt-4.1-nano`
- Azure Speech TTS
- Azure Speech STT / ASR from WAV
- Azure Speech STT / ASR from microphone

In [ ]:
%pip install -q openai azure-cognitiveservices-speech python-dotenv requests ipykernel jupyter

In [1]:

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "https://swedencentral.api.cognitive.microsoft.com/")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
AZURE_OPENAI_GPT51_DEPLOYMENT = os.getenv("AZURE_OPENAI_GPT51_DEPLOYMENT", "gpt-5.1")
AZURE_OPENAI_GPT51_KEY = os.getenv("AZURE_OPENAI_GPT51_KEY", "4.1 key")
AZURE_OPENAI_GPT41_NANO_DEPLOYMENT = os.getenv("AZURE_OPENAI_GPT41_NANO_DEPLOYMENT", "gpt-4.1-nano")
AZURE_OPENAI_GPT41_NANO_KEY = os.getenv("AZURE_OPENAI_GPT41_NANO_KEY", "4.1 Nano key")

AZURE_SPEECH_REGION = os.getenv("AZURE_SPEECH_REGION", "swedencentral")
AZURE_SPEECH_KEY = os.getenv("AZURE_SPEECH_KEY", "speech SDK key")
AZURE_SPEECH_VOICE = os.getenv("AZURE_SPEECH_VOICE", "en-US-Ava:DragonHDLatestNeural")
AZURE_SPEECH_RECOGNITION_LANGUAGE = os.getenv("AZURE_SPEECH_RECOGNITION_LANGUAGE", "en-US")

print("Loaded configuration.")
print("OpenAI endpoint:", AZURE_OPENAI_ENDPOINT)
print("Speech region:", AZURE_SPEECH_REGION)
print("Speech recognition language:", AZURE_SPEECH_RECOGNITION_LANGUAGE)
print("Speech voice:", AZURE_SPEECH_VOICE)


Loaded configuration.
OpenAI endpoint: https://swedencentral.api.cognitive.microsoft.com/
Speech region: swedencentral
Speech recognition language: en-US
Speech voice: en-US-Ava:DragonHDLatestNeural


In [2]:
print(AZURE_OPENAI_GPT51_KEY)

4cd918ce40574a279305aadfba2d2da3


In [4]:

import requests

def check_http_reachability(url: str, timeout: int = 10) -> dict:
    try:
        response = requests.get(url, timeout=timeout)
        return {"ok": True, "status_code": response.status_code, "final_url": response.url}
    except Exception as exc:
        return {"ok": False, "error": str(exc)}

print("GPT-5.1 endpoint check:", check_http_reachability(AZURE_OPENAI_ENDPOINT))
print("GPT-4.1 Nano endpoint check:", check_http_reachability(AZURE_OPENAI_ENDPOINT))


GPT-5.1 endpoint check: {'ok': True, 'status_code': 200, 'final_url': 'https://swedencentral.api.cognitive.microsoft.com/'}
GPT-4.1 Nano endpoint check: {'ok': True, 'status_code': 200, 'final_url': 'https://swedencentral.api.cognitive.microsoft.com/'}


In [5]:

from openai import AzureOpenAI

gpt51_client = AzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_GPT51_KEY,
)

response = gpt51_client.chat.completions.create(
    model=AZURE_OPENAI_GPT51_DEPLOYMENT,
    messages=[
        {"role": "system", "content": "You are a helpful hackathon assistant."},
        {"role": "user", "content": "Give me 3 short ideas for a voicebot hackathon demo."},
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)


1. **Medication Reminder & Check-in Bot**  
   - Voicebot that reminds users to take meds, asks if they’ve taken them, logs responses, and can alert a caregiver if doses are repeatedly missed.  
   - Could use simple voice authentication (“It’s really you?”) and basic health questions (“Any dizziness today?”).

2. **Meeting Summary & Task Extractor**  
   - Join a meeting (or simulate one), then the bot summarizes key points and reads them back.  
   - On voice command (“What are my action items?”), it lists tasks and can send them to email or Slack.

3. **Hands-free Office Concierge**  
   - Voicebot for shared workspaces to book rooms, check desk availability, and submit quick IT or facilities tickets.  
   - Users say things like “Book a room for 3 people at 2 pm” or “Report a broken monitor at Desk 14.”


In [6]:

nano_client = AzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_GPT41_NANO_KEY,
)

response = nano_client.chat.completions.create(
    model=AZURE_OPENAI_GPT41_NANO_DEPLOYMENT,
    messages=[
        {"role": "system", "content": "You are a helpful hackathon assistant."},
        {"role": "user", "content": "Summarize why a small model is useful in a hackathon in 3 bullets."},
    ],
    max_completion_tokens=300,
    temperature=1.0,
    top_p=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
)

print(response.choices[0].message.content)


- Faster Development: Small models require less training time and computational resources, enabling rapid iterations and experimentation during a hackathon.  
- Cost-Effective: They reduce hardware and infrastructure costs, making it feasible to run tasks on limited budgets or hardware.  
- Simplified Deployment: Small models are easier to deploy and optimize on diverse platforms, facilitating quick integration and testing of ideas.


In [ ]:

import azure.cognitiveservices.speech as speechsdk

speech_config = speechsdk.SpeechConfig(subscription=AZURE_SPEECH_KEY, region=AZURE_SPEECH_REGION)
speech_config.speech_synthesis_voice_name = AZURE_SPEECH_VOICE
speech_config.speech_recognition_language = AZURE_SPEECH_RECOGNITION_LANGUAGE

tts_output = Path("speech_output.wav").resolve()
audio_out = speechsdk.audio.AudioOutputConfig(filename=str(tts_output))
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_out)
# audio_config = speechsdk.audio.AudioOutputConfig(filename="speech_output.wav")
# speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

text = "Hi, this is your Azure Voicebot hackathon starter kit."
result = synthesizer.speak_text_async(text).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print("Speech synthesized successfully:", tts_output)
else:
    print("Speech synthesis failed:", result.reason)


Speech synthesized successfully: C:\Users\Q402308\OneDrive - A1 Group\Pictures\Personal\Hackathon\2026\azure_voice_hackathon_starter\v2\speech_output11.wav


## STT / ASR from WAV file
This transcribes the `speech_output.wav` file created in the previous step. You can also replace the path with your own local WAV file.

In [9]:

wav_path = Path("speech_output.wav").resolve()
audio_input = speechsdk.audio.AudioConfig(filename=str(wav_path))
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_input)

result = recognizer.recognize_once_async().get()

if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print("Recognized text from WAV:", result.text)
elif result.reason == speechsdk.ResultReason.NoMatch:
    print("No speech could be recognized from the WAV file.")
else:
    print("WAV transcription failed:", result.reason)
    if result.reason == speechsdk.ResultReason.Canceled:
        print("Details:", result.cancellation_details.error_details)


Recognized text from WAV: Hi this is your Azure Voicebot Hackathon starter kit.


## Optional STT / ASR from microphone
Run this cell only if your local VS Code/Jupyter kernel can access your default microphone.

In [11]:

mic_audio = speechsdk.audio.AudioConfig(use_default_microphone=True)
mic_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=mic_audio)

print("Speak into your default microphone now...")
mic_result = mic_recognizer.recognize_once_async().get()

if mic_result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print("Recognized text from microphone:", mic_result.text)
elif mic_result.reason == speechsdk.ResultReason.NoMatch:
    print("No speech could be recognized from the microphone.")
else:
    print("Microphone transcription failed:", mic_result.reason)
    if mic_result.reason == speechsdk.ResultReason.Canceled:
        print("Details:", mic_result.cancellation_details.error_details)


Speak into your default microphone now...
Recognized text from microphone: Hi, can you hear me?


In [12]:
import threading
import time
from queue import Empty, Queue

speaker_audio = speechsdk.audio.AudioOutputConfig(use_default_speaker=True)
realtime_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=speaker_audio)
realtime_mic_audio = speechsdk.audio.AudioConfig(use_default_microphone=True)
realtime_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=realtime_mic_audio)

realtime_messages = [
    {"role": "system", "content": "You are a concise, helpful voice assistant for a live demo. Keep replies natural, short, and easy to speak out loud."}
]
realtime_queue = Queue()
realtime_stop_event = threading.Event()
realtime_speaking_lock = threading.Lock()
realtime_state_lock = threading.Lock()
realtime_current_request_id = 0
realtime_active_request_id = 0
realtime_speech_started_at = None
realtime_barge_in_seconds = 2.0

def ask_realtime_voicebot(user_text: str) -> str:
    realtime_messages.append({"role": "user", "content": user_text})
    response = gpt51_client.chat.completions.create(
        model=AZURE_OPENAI_GPT51_DEPLOYMENT,
        messages=realtime_messages,
        max_completion_tokens=250,
    )
    reply = (response.choices[0].message.content or "").strip() or "I heard you, but I could not generate a reply."
    realtime_messages.append({"role": "assistant", "content": reply})
    return reply

def clear_realtime_pending_queries() -> None:
    while True:
        try:
            pending_item = realtime_queue.get_nowait()
        except Empty:
            break
        if pending_item != "__STOP__":
            realtime_queue.task_done()
            continue
        realtime_queue.task_done()
        realtime_queue.put("__STOP__")
        break

def interrupt_realtime_output() -> None:
    try:
        realtime_synthesizer.stop_speaking_async().get()
    except Exception:
        pass

def queue_latest_realtime_query(text: str) -> int:
    global realtime_current_request_id, realtime_active_request_id
    with realtime_state_lock:
        realtime_current_request_id += 1
        request_id = realtime_current_request_id
        realtime_active_request_id = request_id
    clear_realtime_pending_queries()
    interrupt_realtime_output()
    realtime_queue.put((request_id, text))
    return request_id

def speak_realtime_reply(text: str) -> None:
    with realtime_speaking_lock:
        result = realtime_synthesizer.speak_text_async(text).get()
        if result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
            details = speechsdk.SpeechSynthesisCancellationDetails.from_result(result)
            print(f"[realtime-tts-error] {details.reason}: {details.error_details}")

def realtime_worker() -> None:
    while not realtime_stop_event.is_set():
        queue_item = realtime_queue.get()
        if queue_item == "__STOP__":
            realtime_queue.task_done()
            break
        request_id, user_text = queue_item
        try:
            print(f"You said: {user_text}")
            reply = ask_realtime_voicebot(user_text)
            with realtime_state_lock:
                if request_id != realtime_active_request_id:
                    print("[interrupted] Skipping an older reply.")
                    print()
                    continue
            print(f"Assistant: {reply}")
            speak_realtime_reply(reply)
            print()
        except Exception as exc:
            print(f"[realtime-error] {exc}")
        finally:
            realtime_queue.task_done()

def on_realtime_recognized(evt):
    global realtime_speech_started_at
    if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
        text = evt.result.text.strip()
        if text:
            realtime_speech_started_at = None
            queue_latest_realtime_query(text)
    elif evt.result.reason == speechsdk.ResultReason.NoMatch:
        realtime_speech_started_at = None
        print("[realtime-stt] No speech could be recognized.")

def on_realtime_recognizing(evt):
    global realtime_speech_started_at
    if evt.result.reason != speechsdk.ResultReason.RecognizingSpeech:
        return
    partial_text = evt.result.text.strip()
    if not partial_text:
        return
    now = time.monotonic()
    if realtime_speech_started_at is None:
        realtime_speech_started_at = now
        return
    if now - realtime_speech_started_at >= realtime_barge_in_seconds:
        interrupt_realtime_output()

def on_realtime_canceled(evt):
    print(f"[realtime-stt-canceled] reason={evt.reason}")
    if evt.reason == speechsdk.CancellationReason.Error:
        print(f"[realtime-stt-error] {evt.error_details}")

realtime_thread = threading.Thread(target=realtime_worker, daemon=True)
realtime_thread.start()
realtime_recognizer.recognizing.connect(on_realtime_recognizing)
realtime_recognizer.recognized.connect(on_realtime_recognized)
realtime_recognizer.canceled.connect(on_realtime_canceled)
realtime_recognizer.start_continuous_recognition_async().get()

print("Realtime voice chat is running. Speak into your default microphone.")
print("If you talk over the assistant for about 2 seconds, it will interrupt and switch to your latest request.")
print("Type stop and press Enter to finish this test cell.")

try:
    while True:
        command = input().strip().lower()
        if command in {"stop", "exit", "quit"}:
            break
finally:
    realtime_stop_event.set()
    realtime_recognizer.stop_continuous_recognition_async().get()
    realtime_queue.put("__STOP__")
    realtime_thread.join(timeout=5)
    print("Realtime voice chat stopped.")


Realtime voice chat is running. Speak into your default microphone.
If you talk over the assistant for about 2 seconds, it will interrupt and switch to your latest request.
Type stop and press Enter to finish this test cell.


You said: Hi, can you hear me?
Assistant: I don’t actually hear sound, but I can see your message just fine. How can I help you today?

You said: Uh, I wanted to know about the different voice boards available out there.
Assistant: Got it—when you say “voice boards,” do you mean:

1. Hardware boards for making devices talk (like text‑to‑speech modules for Arduino/Raspberry Pi)?  
2. Call center / telephony voice boards (PCI/PCIe cards for phone lines and IVR systems)?  
3. Something else, like “sound boards” or gamer voice changers?

If you can tell me which one you’re interested in, I’ll walk through the main options and what they’re good for.

You said: Thank you.
Assistant: You’re welcome!  

If you tell me which kind of voice boards you meant—hardware TTS modules, telephony cards, or something else—I can give you a quick rundown of the main options.
[realtime-error] type object 'SpeechSynthesisCancellationDetails' has no attribute 'from_result'
Realtime voice chat stopped.
